In [1]:
# For processing the timeseries
import pandas as pd, os, datetime
import numpy as np

# For plotting
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
pio.renderers.default = 'notebook'

In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')

nmap_path = '/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess'
ehf_fpath = '/scratch/ng72/ms5578/time_series'
gen_fpath = '/scratch/ng72/ms5578/time_series/nem_generation'

In [3]:
sdate, edate = '2009-07-01','2024-06-30'

In [4]:
gen_details = pd.read_csv(f"{nmap_path}/gen_details.csv",index_col=False)

In [5]:
nmap = pd.read_csv(f"{nmap_path}/nmap.csv",index_col=False)

In [6]:
hw_tseries = pd.read_csv(f"{ehf_fpath}/gen_hw_status.csv",index_col=False)

In [7]:
def process_group(grp, gen_fpath, hw_tseries, start_date=sdate, end_date=edate):
    """
    Processes a single group of generator data.

    Parameters:
        grp (pd.DataFrame): A single group from gen_details (e.g., from groupby('region')).
        gen_fpath (str): Path to directory containing CSV files named by DUID (e.g., DUID.csv).
        hw_tseries (pd.DataFrame): DataFrame with columns 'time', 'DUID', and data to merge.
        start_date (str): Start date for subsetting the time series.
        end_date (str): End date for subsetting the time series.

    Returns:
        pd.DataFrame: The merged result for the group.
    """
    gen_locs = gen_fpath + '/' + grp['DUID'] + ".csv"
    dfs = [pd.read_csv(fp,dtype='object') for fp in gen_locs if os.path.exists(fp)]

    if not dfs:
        return None

    # This is in case of accidental mid-file headers
    dfs = pd.concat(dfs, ignore_index=True)
    header_row = dfs.columns.tolist()
    dfs = dfs[~dfs.apply(lambda row: list(row) == header_row, axis=1)]

    # This is to correct the column types after removing header rows
    dfs['time'] = pd.to_datetime(dfs['time'])
    dfs['TOTALCLEARED'] = dfs['TOTALCLEARED'].astype(float)
    dfs['TOTALMWh'] = dfs['TOTALMWh'].astype(float)
    dfs['AGCSTATUS'] = pd.to_numeric(dfs['AGCSTATUS'], errors='coerce').fillna(0).astype(int)
    
    dfs['time'] = pd.to_datetime(dfs['time'])
    dfs = dfs.set_index('time').sort_index()
    dfs = dfs.loc[start_date:end_date]

    hw_tseries['time'] = pd.to_datetime(hw_tseries['time'])
    hw_tseries = hw_tseries.set_index(['time']).sort_index()
    hw_tseries = hw_tseries.loc[sdate:edate]
    hw_tseries = hw_tseries.reset_index(drop=True).set_index(['DUID','time']).sort_index()

    merged = pd.merge_asof(
        dfs.sort_values(by=['time', 'DUID']),
        hw_tseries.sort_values(by=['time', 'DUID']),
        by='DUID',
        on='time',
        tolerance=pd.Timedelta("1d"),
        direction='nearest'
    )
    merged = merged.reset_index(drop=True)

    return merged

In [20]:
def clean_df(df,hw=False,gen_details=gen_details,AGC=False):
    df = df.merge(gen_details[['DUID', 'reg_cap_generation_mw']], on='DUID', how='left')
    df['reg_cap_generation_mw'] = df['reg_cap_generation_mw'].astype('float')
    df['cap_norm'] = df['TOTALMWh']/df['reg_cap_generation_mw']
    df = df[~((df['cap_norm'] > 1.2) | (df['cap_norm'] <0)) ]
    df = df.drop(['cap_norm','reg_cap_generation_mw'],axis=1)

    df_sliced = df.loc[(df['time'].dt.month >= 11)| (df['time'].dt.month <= 3)]

    if hw == True:
        df_sliced = df_sliced[df_sliced['EHF_flag'] == 1]

    if AGC == True:
        df_sliced = df_sliced[df_sliced['AGCSTATUS'] == 1]
    
    return df_sliced

In [18]:
df = process_group(gen_details, gen_fpath,hw_tseries,sdate,edate)

In [21]:
df = clean_df(df)

In [51]:
agg_func = {'TOTALMWh':'sum','EHF_flag':'first'}
agg_df = df.set_index('time').groupby(['DUID', pd.Grouper(freq='1D')]).agg(agg_func).reset_index()
agg_df = agg_df.groupby(['DUID','EHF_flag']).mean().reset_index().drop(['time'],axis=1)

In [52]:
agg_df = agg_df.pivot(index='DUID', columns='EHF_flag', values='TOTALMWh')
agg_df['diff'] = agg_df[1.0] - agg_df[0.0]

In [ ]:
fig = px.scatter_map(
    df_plot,
    lat='lat',
    lon='lon',
    color='cluster',
    color_discrete_sequence=px.colors.qualitative.Light24,
    hover_name='DUID',
    hover_data=['time','fuel_source_primary'],
    zoom=5,
    map_style='carto-positron',
    title='Heatwave Clusters by DBSCAN'
)

fig.show()